# 🤖 PyCaret AutoML 실습

> **학습 목표**: PyCaret을 사용하여 자동으로 최적의 머신러닝 모델을 찾고 학습합니다.

---

## 📋 학습 내용

1. ✅ PyCaret 기본 사용법
2. ✅ 자동 전처리 및 피처 엔지니어링
3. ✅ 18+ 모델 자동 비교
4. ✅ 하이퍼파라미터 튜닝
5. ✅ Ensemble 모델 생성
6. ✅ SHAP 통합 (모델 해석)
7. ✅ 모델 저장 및 배포

**소요 시간**: 약 45분  
**난이도**: ⭐⭐ (중급)  
**사전 지식**: 머신러닝 기초, scikit-learn 사용 경험

---

## 🔧 Step 1: 라이브러리 Import

In [ ]:
# 데이터 분석
import pandas as pd
import numpy as np

# PyCaret
try:
    from pycaret.classification import *
    import pycaret
    print(f"✅ PyCaret 버전: {pycaret.__version__}")
except ImportError:
    print("❌ PyCaret 설치 필요: pip install pycaret")
    print("   또는: pip install pycaret[full]")

# 시각화
import matplotlib.pyplot as plt
import seaborn as sns

# 유틸리티
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트
plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (14, 6)

print("\n✅ 라이브러리 로드 완료!")

## 📂 Step 2: 데이터 로드 및 준비

In [ ]:
# KAMP CNC 데이터 로드
data_path = Path('../data/sample_kamp_cnc.csv')

# 데이터 로드 (인코딩 자동 처리)
def load_data(file_path):
    encodings = ['utf-8-sig', 'cp949', 'utf-8']
    for encoding in encodings:
        try:
            df = pd.read_csv(file_path, encoding=encoding)
            print(f"✅ 데이터 로드 성공! (인코딩: {encoding})")
            return df
        except:
            continue
    raise ValueError("❌ 데이터 로드 실패")

df = load_data(data_path)
print(f"\n📊 데이터 크기: {df.shape[0]:,}행 × {df.shape[1]}열")
print(f"💾 메모리 사용량: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# 데이터 미리보기
df.head()

In [ ]:
# 타겟 변수 생성 (이진 분류)
# 수치형 변수 확인
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"📊 수치형 변수: {len(numeric_cols)}개")
print(f"   {numeric_cols[:10]}...")

# 타겟 생성: 첫 번째 수치 컬럼의 중앙값 기준
if len(numeric_cols) > 0:
    target_col = numeric_cols[0]
    threshold = df[target_col].median()
    
    df['quality_class'] = (df[target_col] > threshold).astype(int)
    
    print(f"\n🎯 타겟 변수 생성:")
    print(f"   - 기준 컬럼: {target_col}")
    print(f"   - 임계값: {threshold:.4f}")
    print(f"   - 클래스 분포:")
    print(df['quality_class'].value_counts())
    
    # 피처 컬럼 (타겟 제외)
    feature_cols = [col for col in numeric_cols if col != target_col]
    
    # 최종 데이터셋
    data = df[feature_cols + ['quality_class']].copy()
    
    # 결측치 제거
    data = data.dropna()
    print(f"\n✅ 분석용 데이터셋: {data.shape[0]:,}행 × {data.shape[1]}열")
else:
    print("⚠️ 수치형 변수를 찾을 수 없습니다.")
    data = df

## 🤖 Step 3: PyCaret Setup (자동 전처리)

### PyCaret이 자동으로 수행하는 작업:
- 결측치 처리
- 범주형 변수 인코딩
- 스케일링/정규화
- 피처 엔지니어링
- 데이터 분할 (Train/Test)

In [ ]:
# PyCaret Setup
print("🔄 PyCaret Setup 실행 중...\n")

try:
    clf = setup(
        data=data,
        target='quality_class',
        session_id=42,
        train_size=0.8,
        normalize=True,
        transformation=True,
        ignore_low_variance=True,
        remove_multicollinearity=True,
        multicollinearity_threshold=0.9,
        n_jobs=-1,
        silent=True,
        verbose=False
    )
    
    print("\n✅ PyCaret Setup 완료!")
    print("\n📊 전처리 요약:")
    print(f"   - Train 크기: {int(len(data) * 0.8):,}개")
    print(f"   - Test 크기: {int(len(data) * 0.2):,}개")
    print(f"   - 피처 수: {len(feature_cols)}개")
    print(f"   - 정규화: ✅")
    print(f"   - 다중공선성 제거: ✅")
    
except Exception as e:
    print(f"❌ Setup 실패: {e}")
    print("   PyCaret 설치: pip install pycaret")

## 📊 Step 4: 모델 자동 비교

PyCaret은 18개 이상의 모델을 자동으로 학습하고 비교합니다.

In [ ]:
# 모델 자동 비교 (상위 5개 선택)
print("🔄 모델 비교 실행 중 (18+ 알고리즘)...\n")

try:
    # 모델 비교
    best_models = compare_models(n_select=5, sort='Accuracy', turbo=False)
    
    print("\n✅ 모델 비교 완료!")
    print("\n📊 상위 5개 모델 선택됨")
    
except Exception as e:
    print(f"⚠️ 모델 비교 실패: {e}")

In [ ]:
# 비교 결과 테이블 추출 및 시각화
try:
    # 결과 테이블 가져오기
    comparison_df = pull()
    
    print("📊 모델 비교 결과 (상위 10개):")
    print(comparison_df.head(10).to_string())
    
    # 상위 10개 모델 시각화
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Accuracy 비교
    top10 = comparison_df.head(10)
    axes[0].barh(range(len(top10)), top10['Accuracy'], color='skyblue')
    axes[0].set_yticks(range(len(top10)))
    axes[0].set_yticklabels(top10.index, fontsize=10)
    axes[0].set_xlabel('Accuracy')
    axes[0].set_title('상위 10개 모델 - Accuracy', fontsize=12, fontweight='bold')
    axes[0].invert_yaxis()
    axes[0].grid(alpha=0.3)
    
    # F1 Score 비교
    axes[1].barh(range(len(top10)), top10['F1'], color='coral')
    axes[1].set_yticks(range(len(top10)))
    axes[1].set_yticklabels(top10.index, fontsize=10)
    axes[1].set_xlabel('F1 Score')
    axes[1].set_title('상위 10개 모델 - F1 Score', fontsize=12, fontweight='bold')
    axes[1].invert_yaxis()
    axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
except Exception as e:
    print(f"⚠️ 결과 시각화 실패: {e}")

## ⚙️ Step 5: 모델 튜닝

In [ ]:
# 최상위 모델 하이퍼파라미터 튜닝
print("🔄 하이퍼파라미터 튜닝 중...\n")

try:
    # 첫 번째 best 모델 튜닝
    tuned_model = tune_model(
        best_models[0],
        optimize='Accuracy',
        n_iter=50,
        search_library='scikit-learn',
        search_algorithm='random',
        verbose=False
    )
    
    print("\n✅ 튜닝 완료!")
    
    # 튜닝 결과
    tuned_results = pull()
    print("\n📊 튜닝 결과:")
    print(tuned_results.to_string())
    
except Exception as e:
    print(f"⚠️ 튜닝 실패: {e}")
    tuned_model = best_models[0]

## 📈 Step 6: 모델 평가 및 시각화

In [ ]:
# Confusion Matrix
try:
    print("📊 Confusion Matrix 생성 중...")
    plot_model(tuned_model, plot='confusion_matrix', display_format='streamlit')
    print("✅ Confusion Matrix 생성 완료")
except Exception as e:
    print(f"⚠️ Confusion Matrix 생성 실패: {e}")

In [ ]:
# AUC-ROC Curve
try:
    print("📊 AUC-ROC Curve 생성 중...")
    plot_model(tuned_model, plot='auc', display_format='streamlit')
    print("✅ AUC-ROC Curve 생성 완료")
except Exception as e:
    print(f"⚠️ AUC-ROC 생성 실패: {e}")

In [ ]:
# Feature Importance
try:
    print("📊 Feature Importance 생성 중...")
    plot_model(tuned_model, plot='feature', display_format='streamlit')
    print("✅ Feature Importance 생성 완료")
except Exception as e:
    print(f"⚠️ Feature Importance 생성 실패: {e}")

## 🎭 Step 7: Ensemble 모델 생성

In [ ]:
# Ensemble: Bagging
print("🔄 Ensemble 모델 생성 중...\n")

try:
    # Bagging Ensemble
    bagged_model = ensemble_model(tuned_model, method='Bagging', n_estimators=10)
    
    # Boosting Ensemble
    boosted_model = ensemble_model(tuned_model, method='Boosting', n_estimators=10)
    
    print("\n✅ Ensemble 모델 생성 완료!")
    
    # 성능 비교
    print("\n📊 모델 성능 비교:")
    print(f"   - 기본 모델: {comparison_df.iloc[0]['Accuracy']:.4f} (Accuracy)")
    
    bagged_results = pull()
    print(f"   - Bagged 모델: {bagged_results['Accuracy'].mean():.4f} (Accuracy)")
    
except Exception as e:
    print(f"⚠️ Ensemble 생성 실패: {e}")
    bagged_model = tuned_model

## 🔍 Step 8: SHAP 통합 (모델 해석)

PyCaret은 SHAP과 자동 통합되어 모델 해석 가능

In [ ]:
# SHAP Interpretation
try:
    print("🔍 SHAP 모델 해석 중...")
    interpret_model(tuned_model)
    print("\n✅ SHAP 해석 완료!")
    print("\n💡 자세한 SHAP 분석은 '03_shap_explainer.ipynb' 참조")
except Exception as e:
    print(f"⚠️ SHAP 해석 실패: {e}")
    print("   일부 모델은 SHAP을 지원하지 않을 수 있습니다.")

## 💾 Step 9: 모델 저장 및 예측

In [ ]:
# 출력 디렉토리 생성
output_dir = Path('../outputs')
output_dir.mkdir(exist_ok=True)

# 모델 저장
try:
    model_path = output_dir / '02_pycaret_best_model'
    save_model(bagged_model, str(model_path))
    print(f"✅ 모델 저장: {model_path}.pkl")
except Exception as e:
    print(f"⚠️ 모델 저장 실패: {e}")

# 비교 결과 저장
try:
    comparison_file = output_dir / '02_model_comparison.csv'
    comparison_df.to_csv(comparison_file, index=True, encoding='utf-8-sig')
    print(f"✅ 비교 결과 저장: {comparison_file}")
except Exception as e:
    print(f"⚠️ 비교 결과 저장 실패: {e}")

print("\n🎉 PyCaret AutoML 완료!")

In [ ]:
# 예측 예시 (새로운 데이터)
try:
    # 테스트 데이터로 예측
    test_sample = data.drop('quality_class', axis=1).sample(10)
    predictions = predict_model(bagged_model, data=test_sample)
    
    print("📊 예측 결과 (샘플 10개):")
    print(predictions[['prediction_label', 'prediction_score']].to_string())
    
except Exception as e:
    print(f"⚠️ 예측 실패: {e}")

---

## 🎯 학습 정리

### ✅ 완료한 내용
1. PyCaret Setup으로 자동 전처리 수행
2. 18개 이상 모델 자동 비교 및 평가
3. 최상위 모델 하이퍼파라미터 튜닝
4. Bagging/Boosting Ensemble 모델 생성
5. Confusion Matrix, AUC, Feature Importance 시각화
6. SHAP 통합으로 모델 해석
7. 모델 저장 및 배포 준비

### 💡 핵심 인사이트

- **자동화의 힘**:
  - 전통적 방법: 수백 줄 코드, 수동 전처리, 개별 모델 학습
  - PyCaret: 10줄 코드로 18+ 모델 자동 비교, 최적화
  - 시간 절약: 수 시간 → 수 분

- **PyCaret의 장점**:
  - Low-code ML: 최소한의 코드로 전체 파이프라인 구축
  - 자동 전처리: 결측치, 인코딩, 스케일링 자동 처리
  - 모델 비교: 18+ 알고리즘 동시 비교
  - 하이퍼파라미터 튜닝: Grid/Random/Bayesian Search
  - Ensemble: Bagging, Boosting, Stacking 자동 생성
  - 배포: API, Docker, ONNX 변환 지원

- **PyCaret vs scikit-learn**:
  - scikit-learn: 세밀한 제어, 커스터마이징
  - PyCaret: 빠른 프로토타이핑, 자동화
  - 병행 사용: PyCaret로 빠른 실험 → scikit-learn으로 정교화

- **SHAP 통합**:
  - `interpret_model()`로 자동 SHAP 분석
  - 모델 블랙박스 문제 해결
  - 자세한 분석: `03_shap_explainer.ipynb` 참조

- **실무 활용**:
  - 빠른 POC (Proof of Concept)
  - 베이스라인 모델 구축
  - 피처 중요도 분석
  - 모델 선정 가이드

- **주의사항**:
  - 대용량 데이터: 메모리 부족 가능 (샘플링 필요)
  - 복잡한 피처 엔지니어링: 수동 전처리 후 사용
  - 프로덕션 배포: 추가 최적화 필요

### 📚 다음 단계
- **03_shap_explainer.ipynb**: SHAP 심화 분석 (PyCaret 모델 연동)
- **모델 배포**: FastAPI, Docker, Streamlit 연동
- **AutoML 고도화**: H2O.ai, Auto-sklearn 비교

### 🔗 참고 자료
- [PyCaret 공식 문서](https://pycaret.gitbook.io/docs/)
- [PyCaret GitHub](https://github.com/pycaret/pycaret)
- [PyCaret 튜토리얼](https://pycaret.org/tutorial/)

---

*제조AI 교육 v12 Enhanced | Part 1-1 | 2025.02*